# DEH 30-Day PySpark Challenge
## Day 13 — Joins: inner, left, right, full outer

**Instructions:**
1. Click `File → Save a copy in Drive` before editing
2. Run each cell in order using `Shift + Enter`
3. Complete the assignment cells at the bottom

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder
    .appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

In [ ]:
orders_schema = StructType([
    StructField("order_id",       StringType(),  True),
    StructField("customer_id",    StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("order_date",     DateType(),    True),
    StructField("quantity",       IntegerType(), True),
    StructField("unit_price",     DoubleType(),  True),
    StructField("discount_pct",   IntegerType(), True),
    StructField("status",         StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("region",         StringType(),  True)
])

customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name",  StringType(), True),
    StructField("last_name",   StringType(), True),
    StructField("email",       StringType(), True),
    StructField("city",        StringType(), True),
    StructField("state",       StringType(), True),
    StructField("country",     StringType(), True),
    StructField("signup_date", DateType(),   True),
    StructField("segment",     StringType(), True)
])

products_schema = StructType([
    StructField("product_id",     StringType(), True),
    StructField("product_name",   StringType(), True),
    StructField("category",       StringType(), True),
    StructField("sub_category",   StringType(), True),
    StructField("unit_price",     DoubleType(), True),
    StructField("cost_price",     DoubleType(), True),
    StructField("supplier",       StringType(), True),
    StructField("stock_quantity", IntegerType(),True)
])

orders_df    = spark.read.option("header","true").schema(orders_schema).csv(f"s3a://{BUCKET}/data/orders.csv")
customers_df = spark.read.option("header","true").schema(customers_schema).csv(f"s3a://{BUCKET}/data/customers.csv")
products_df  = spark.read.option("header","true").schema(products_schema).csv(f"s3a://{BUCKET}/data/products.csv")

print(f"Orders: {orders_df.count()} | Customers: {customers_df.count()} | Products: {products_df.count()}")

## Step 5 — inner join

In [ ]:
result = orders_df.join(customers_df, on="customer_id", how="inner")

print(f"Orders     : {orders_df.count()}")
print(f"Customers  : {customers_df.count()}")
print(f"After join : {result.count()}")

result.select(
    "order_id", "customer_id", "first_name", "last_name", "unit_price", "status"
).show(5)

## Step 6 — left join

In [ ]:
result = orders_df.join(customers_df, on="customer_id", how="left")

print(f"Orders     : {orders_df.count()}")
print(f"After join : {result.count()}")

# Orders with no matching customer
unmatched = result.filter(F.col("first_name").isNull())
print(f"Orders with no customer record: {unmatched.count()}")

## Step 7 — right join

In [ ]:
result = orders_df.join(customers_df, on="customer_id", how="right")

print(f"Customers  : {customers_df.count()}")
print(f"After join : {result.count()}")

# Customers with no orders
no_orders = result.filter(F.col("order_id").isNull())
print(f"Customers with no orders: {no_orders.count()}")
no_orders.select("customer_id", "first_name", "last_name").show()

## Step 8 — full outer join

In [ ]:
result = orders_df.join(customers_df, on="customer_id", how="outer")

print(f"After full outer join: {result.count()}")

orders_only   = result.filter(F.col("first_name").isNull())
customers_only = result.filter(F.col("order_id").isNull())

print(f"Orders with no customer : {orders_only.count()}")
print(f"Customers with no orders: {customers_only.count()}")

## Step 9 — Handling duplicate column names after join

In [ ]:
# Both orders and products have unit_price — fix with aliases
result = orders_df.join(products_df, on="product_id", how="inner") \
    .select(
        orders_df["order_id"],
        orders_df["customer_id"],
        products_df["product_name"],
        orders_df["unit_price"].alias("order_price"),
        products_df["unit_price"].alias("list_price"),
        orders_df["quantity"],
        orders_df["status"]
    )

result.show(5)

---
## Assignment — Day 14

---

### Task 1
Join `orders.csv` with `customers.csv` using an **inner join** on `customer_id`.  
Select `order_id`, `first_name`, `last_name`, `unit_price`, and `region`.  
Show the first 5 rows. What is the row count?

In [ ]:
# Task 1 — Write your code here


### Task 2
Join `orders.csv` with `customers.csv` using a **left join**.  
How many orders have no matching customer?  
Filter for those rows and show them.

In [ ]:
# Task 2 — Write your code here


### Task 3
Join `orders.csv` with `products.csv` using an **inner join** on `product_id`.  
Both DataFrames have a `unit_price` column — handle the duplicate using aliases:  
`orders_df["unit_price"].alias("order_price")` and `products_df["unit_price"].alias("list_price")`.  
Show the result.

In [ ]:
# Task 3 — Write your code here


### Task 4
Perform a **full outer join** between `customers.csv` and `orders.csv`.  
Find customers who have never placed an order.  
How many are there?

In [ ]:
# Task 4 — Write your code here


---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*